# TrackMind
## Advanced Machine Learning Group Project

What's in the pipeline:

- TSI auto-downloads from EUR-Lex (unchanged, still needs the User-Agent header)
- French NNTR (france_nntr.pdf) is attempted via Légifrance but likely returns HTML - the script detects this and tells you to download manually instead of crashing
- SpecDoc is checked separately since it's a file you generated yourself

**On the Légifrance URL** - their site doesn't serve raw PDFs from a stable direct link the way EUR-Lex does. The download attempt is included as a convenience, but realistically you'll need to do it manually. Go to legifrance.gouv.fr/loda/id/JORFTEXT000025582663, click the PDF export button in the top right, and save as docs/france_nntr.pdf. The script detects whether what it downloaded is actually a valid PDF and warns you clearly if not.

**On the chunking** - the French arrêté uses "Article 49" style headers so the pattern now catches both 4.2.3.1 Title (TSI structure) and Article 49 (French arrêté structure), with a fixed-size fallback if neither produces enough chunks.

**One thing to watch** - the full LOC&PAS TSI is large so the EUR-Lex download takes 20-30 seconds and embedding all its chunks will take a few minutes. The French arrêté is shorter so that goes faster. Both are expected.

The cross-lingual test at the end is the key thing to paste back here - if bge-m3 is working correctly, an English query about "passenger door single agent operation" should surface the French Article 49 NF F31-054 clause from the NNTR collection. That output is the evidence that the multilingual moat argument is real and not just a claim.

In [1]:
import fitz  # PyMuPDF
import chromadb
import re
import os
import requests
from sentence_transformers import SentenceTransformer
from trackmind_chunker import chunk_french_nntr, chunk_tsi_loc_pas, chunk_generic

# ── MODEL ─────────────────────────────────────────────────────────────────────
print("Loading bge-m3 model (first run downloads ~2.5GB, cached after)...")
model = SentenceTransformer('BAAI/bge-m3')
print("Model ready.\n")

# ── CHROMA ────────────────────────────────────────────────────────────────────
chroma = chromadb.PersistentClient(path="./chroma_db")
tsi_col  = chroma.get_or_create_collection("tsi_loc_pas")
nntr_col = chroma.get_or_create_collection("nntr_france")
spec_col = chroma.get_or_create_collection("spec_doc")

# ── DOCUMENT SOURCES ──────────────────────────────────────────────────────────
# TSI: LOC&PAS Commission Regulation (EU) No 1302/2014 - consolidated to 27 April 2025.
#      Auto-downloaded on first run. User-Agent header required - EUR-Lex
#      returns 403 without it.
# NNTR: Arrete du 19 mars 2012 - current active French national rule for
#       rolling stock on the RFN. Article 49 covers passenger door requirements
#       including NF F31-054 for CAS-operated trains.
#       Source: Legifrance (official French government legal database).
#       NOTE: Legifrance may return HTML instead of PDF. ingest.py detects
#       this and tells you clearly. If that happens, go to:
#       legifrance.gouv.fr/loda/id/JORFTEXT000025582663
#       click the PDF export button top-right, save as docs/france_nntr.pdf.
#
# SPEC: SpecDoc.pdf - IberRail IB-EMU-450 fictional technical specification
#       Rev. 4.0. Contains both French compliance conflicts. Must be in docs/.

TSI_SOURCES = {
    "docs/loc_pas_tsi.pdf":
        "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:02014R1302-20250427",
    "docs/france_nntr.pdf":
        "https://www.legifrance.gouv.fr/download/pdf?id=p1BvMpODX6t9do70eAlbQHWbx9QVFqFHyQm72Iregis=",
}

# ── DOWNLOAD ──────────────────────────────────────────────────────────────────
def download_documents():
    os.makedirs("docs", exist_ok=True)
    for local_path, url in TSI_SOURCES.items():
        if os.path.exists(local_path):
            print(f"  Already present: {local_path} - skipping download.")
            continue
        print(f"  Downloading: {url}")
        try:
            r = requests.get(url, timeout=60, headers={
                "User-Agent": "Mozilla/5.0 (compatible; TrackMind-AI/1.0)",
                "Accept": "application/pdf,text/html,*/*",
            })
            r.raise_for_status()

            # Detect if Legifrance returned HTML instead of PDF
            content_type = r.headers.get("Content-Type", "")
            if "html" in content_type or r.content[:4] != b'%PDF':
                print(f"  WARNING: {local_path} download returned HTML, not a PDF.")
                print(f"  Manual download required:")
                print(f"  Go to: legifrance.gouv.fr/loda/id/JORFTEXT000025582663")
                print(f"  Click the PDF export button (top-right), save as {local_path}")
                continue

            with open(local_path, "wb") as f:
                f.write(r.content)
            size_kb = os.path.getsize(local_path) // 1024
            print(f"  Saved {local_path} ({size_kb} KB)")

        except requests.RequestException as e:
            print(f"  Download failed: {e}")
            print(f"  Manual fallback: download from {url}")
            print(f"  Save as: {local_path}\n")

def check_spec_doc():
    if not os.path.exists("docs/SpecDoc.pdf"):
        print("  MISSING: docs/SpecDoc.pdf")
        print("  Copy SpecDoc.pdf into the docs/ folder and rerun.\n")
        return False
    print("  docs/SpecDoc.pdf present.")
    return True

# ── EMBED ─────────────────────────────────────────────────────────────────────
def embed(text):
    # bge-m3 handles French, English, Spanish, Portuguese natively.
    # Cross-lingual: English query retrieves French document chunks correctly.
    text = text[:6000]  # prevents oversized chunks from crashing bge-m3
    return model.encode(text, normalize_embeddings=True).tolist()


# ── INGEST ────────────────────────────────────────────────────────────────────
def ingest(collection, chunks):
    if not chunks:
        print(f"  No chunks to ingest for '{collection.name}' - skipping.")
        return

    existing = collection.count()
    if existing > 0:
        print(f"  '{collection.name}' already has {existing} chunks - skipping.")
        print(f"  To re-ingest: delete ./chroma_db and rerun.\n")
        return

    print(f"  Embedding {len(chunks)} chunks into '{collection.name}'...")
    print(f"  Estimated time: {len(chunks) // 15}-{len(chunks) // 8} minutes on CPU.")
    for i, chunk in enumerate(chunks):
        collection.add(
            ids=       [chunk["id"]],
            embeddings=[embed(chunk["text"])],
            documents= [chunk["text"]],
            metadatas= [chunk["metadata"]]
        )
        if (i + 1) % 10 == 0:
            print(f"    {i + 1}/{len(chunks)} chunks done...")
    print(f"  Done.\n")

# ── RETRIEVAL TEST ────────────────────────────────────────────────────────────
def test_retrieval(query, n=3):
    """
    Cross-lingual test: English query retrieving from French NNTR document.
    bge-m3 should surface the NF F31-054 / Article 49 clause even though
    the source text is in French. This is the live proof of the multilingual
    moat argument - paste the output here to verify before moving on.
    """
    print(f"\n  Test query (English): '{query}'")
    q_emb = embed(query)
    for col, name, lang in [
        (tsi_col,  "TSI  ", "en"),
        (nntr_col, "NNTR ", "fr"),
        (spec_col, "SPEC ", "en"),
    ]:
        if col.count() == 0:
            print(f"  [{name}] Empty - not yet ingested.")
            continue
        results = col.query(query_embeddings=[q_emb], n_results=n)
        print(f"\n  [{name}] Top result (source language: {lang}):")
        if results["documents"][0]:
            doc  = results["documents"][0][0]
            meta = results["metadatas"][0][0]
            print(f"    Article:  {meta.get('article', 'unknown')}")
            print(f"    Language: {meta.get('language', 'unknown')}")
            print(f"    Preview:  {doc[:250].strip()}...")
        else:
            print(f"    No results.")

# ── MAIN ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Quick model sanity check before committing to full ingestion.
    # Should print 1024. If it crashes, fix the model before proceeding.
    print("=== MODEL SANITY CHECK ===")
    test_emb = embed("test query for door obstacle detection")
    print(f"  Embedding dimensions: {len(test_emb)} (expected 1024)")
    if len(test_emb) != 1024:
        print("  WARNING: unexpected embedding size - check bge-m3 installation.")
    else:
        print("  Model working correctly.\n")

    print("=== STEP 1: DOWNLOADING DOCUMENTS ===")
    download_documents()

    print("\n=== STEP 2: CHECKING SPECDOC ===")
    spec_ready = check_spec_doc()

    print("\n=== STEP 3: INGESTING TSI (English) ===")
    if os.path.exists("docs/loc_pas_tsi.pdf"):
        tsi_chunks = chunk_tsi_loc_pas("docs/loc_pas_tsi.pdf", doc_type="TSI_LOC_PAS")
        ingest(tsi_col, tsi_chunks)
    else:
        print("  SKIPPING: docs/loc_pas_tsi.pdf not found.")

    print("=== STEP 4: INGESTING FRANCE NNTR (French) ===")
    if os.path.exists("docs/france_nntr.pdf"):
        nntr_chunks = chunk_french_nntr("docs/france_nntr.pdf", doc_type="NNTR_FRANCE")
        ingest(nntr_col, nntr_chunks)
    else:
        print("  SKIPPING: docs/france_nntr.pdf not found.")
        print("  Download manually from legifrance.gouv.fr/loda/id/JORFTEXT000025582663\n")

    if spec_ready:
        print("=== STEP 5: INGESTING SPECDOC (English) ===")
        spec_chunks = chunk_generic("docs/SpecDoc.pdf", doc_type="SPEC_DOC", language="en")
        ingest(spec_col, spec_chunks)
    else:
        print("=== STEP 5: SKIPPING SPECDOC (file missing) ===\n")

    print("=== FINAL CHUNK COUNTS ===")
    print(f"  TSI:         {tsi_col.count()}")
    print(f"  France NNTR: {nntr_col.count()}")
    print(f"  Spec:        {spec_col.count()}")

    # Cross-lingual retrieval test.
    # Paste this output to verify bge-m3 is crossing the language boundary.
    # The NNTR top result should surface Art. 49 or passenger-door safety language.
    # NF F31-054 test details come from SpecDoc, not the NNTR PDF.
    print("\n=== CROSS-LINGUAL RETRIEVAL TEST ===")
    test_retrieval("passenger access doors closing and locking obstacle detection maximum force")
    test_retrieval("French national rule passenger doors rolling stock Article 49")
    test_retrieval("single agent operation passenger door requirements CAS NF F31-054")
    test_retrieval("door obstacle detection kinetic energy five height positions")

/opt/anaconda3/envs/aml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading bge-m3 model (first run downloads ~2.5GB, cached after)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 22360.93it/s]


Model ready.

=== MODEL SANITY CHECK ===
  Embedding dimensions: 1024 (expected 1024)
  Model working correctly.

=== STEP 1: DOWNLOADING DOCUMENTS ===
  Already present: docs/loc_pas_tsi.pdf - skipping download.
  Already present: docs/france_nntr.pdf - skipping download.

=== STEP 2: CHECKING SPECDOC ===
  docs/SpecDoc.pdf present.

=== STEP 3: INGESTING TSI (English) ===
  Reading docs/loc_pas_tsi.pdf...
  Extracted 516 chunks from TSI.
  'tsi_loc_pas' already has 516 chunks - skipping.
  To re-ingest: delete ./chroma_db and rerun.

=== STEP 4: INGESTING FRANCE NNTR (French) ===
  Reading docs/france_nntr.pdf...
  Extracted 153 chunks from NNTR.
  'nntr_france' already has 153 chunks - skipping.
  To re-ingest: delete ./chroma_db and rerun.

=== STEP 5: INGESTING SPECDOC (English) ===
  Reading docs/SpecDoc.pdf (generic chunker)...
  Extracted 26 chunks (generic).
  'spec_doc' already has 26 chunks - skipping.
  To re-ingest: delete ./chroma_db and rerun.

=== FINAL CHUNK COUNTS ===

In [2]:
import os
expected = ["loc_pas_tsi.pdf", "france_nntr.pdf", "SpecDoc.pdf"]
for f in expected:
    path = f"docs/{f}"
    exists = os.path.exists(path)
    size = os.path.getsize(path) // 1024 if exists else 0
    print(f"  {'OK' if exists else 'MISSING'} {path} ({size} KB)")

  OK docs/loc_pas_tsi.pdf (3504 KB)
  OK docs/france_nntr.pdf (890 KB)
  OK docs/SpecDoc.pdf (47 KB)
